# Project AoU (ACAF) onto the 1000G PCs -- Hail version

Replaces the plink2-against-`plink_bed`/`pgen` extraction in `submit_pca_r1.ipynb`'s "Project AoU data onto the 1000G PCs" section, which hung indefinitely over the gcsfuse-mounted filesystem regardless of file format. Hail reads the ACAF MatrixTable directly via `gs://` (its own GCS-native I/O, not the FUSE mount), which is what that section actually needed.

Run `submit_pca_r1.ipynb` through the "PCA" cell first -- this notebook picks up from there (`PRUNE_PREFIX`, `PCA_PREFIX`) and only replaces the ACAF extraction + projection steps.

## Setup

Requires a Hail-enabled environment (see resource notes -- this is an environment/application *type* choice on Verily Workbench, not something to pip-install on top of a plain Jupyter environment).

In [ ]:
import hail as hl
import os

hl.init(default_reference="GRCh38")

## Paths

`ACAF_MT_PATH` is a native `gs://` URI, not the `~/workspace/...` POSIX mount -- Hail's `hl.read_matrix_table` reads GCS directly.

`WORKSPACE_BUCKET_GS`: your own workspace bucket, as a `gs://` URI (for the Hail checkpoint below). Confirm the exact value for this Hail environment -- it's whatever your own prior working notebook had as `KGP_BUCKET`.

In [ ]:
BUCKET_DIR = os.path.expanduser(
    "~/workspace/Data from All of Us Controlled Tier /shared-env-pilot/data/01_ancestry_filtering"
)
PRUNE_PREFIX = f"{BUCKET_DIR}/all_pruned"
OUT_PREFIX = f"{BUCKET_DIR}/1kg_all_hm3"      # post-QC, pre-pruning bed/bim/fam -- has real CHR/POS/A1/A2 columns
PCA_PREFIX = f"{BUCKET_DIR}/round1_pca/1kg_all_pca"

ACAF_MT_PATH = "gs://vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/acaf_threshold/multiMT/hail.mt"

# PLACEHOLDER -- set to your actual workspace bucket gs:// URI
WORKSPACE_BUCKET_GS = "gs://PLACEHOLDER"
CHECKPOINT_MT = f"{WORKSPACE_BUCKET_GS}/01_ancestry_filtering/round1_pca/acaf_hm3_checkpoint.mt"

LOCAL_WORK_DIR = os.path.expanduser("~/scratch_acaf_hail")
os.makedirs(LOCAL_WORK_DIR, exist_ok=True)
EXPORT_PREFIX = f"{LOCAL_WORK_DIR}/acaf_hm3_subset"

PROJECT_PREFIX = f"{BUCKET_DIR}/round1_pca/acaf_hm3_subset"
PROJECTED_OUT = f"{BUCKET_DIR}/round1_pca/acaf_projected"
os.makedirs(os.path.dirname(PROJECT_PREFIX), exist_ok=True)

## Build the HM3 locus filter

Rather than parsing `all_pruned.prune.in`'s `chr:pos:ref:alt` ID strings directly, filter `OUT_PREFIX.bim` (real `CHR ID CM POS A1 A2` columns, pre-pruning but post-HM3-QC) down to just the pruned IDs -- gives Hail a proper bim-shaped table to import, matching the working pattern.

Chromosome naming: main chromosomes come through as `'1'`, `'2'`, ... and alt contigs already as `'chr1_KI270759v1_alt'` etc. -- conditionally add the `chr` prefix so both map to valid GRCh38 contigs, rather than assuming one fixed prefix (which was my earlier, wrong, assumption).

In [ ]:
%%bash -s "$PRUNE_PREFIX" "$OUT_PREFIX"
set -e
PRUNE_PREFIX=$1
OUT_PREFIX=$2

awk 'NR==FNR{keep[$1]=1; next} keep[$2]' "${PRUNE_PREFIX}.prune.in" "${OUT_PREFIX}.bim" > "${PRUNE_PREFIX}.pruned.bim"
wc -l "${PRUNE_PREFIX}.pruned.bim"


In [ ]:
hm3_bim = hl.import_table(
    f"{PRUNE_PREFIX}.pruned.bim",
    no_header=True,
    types={"f3": hl.tint32},
)
contig_38 = hl.if_else(hm3_bim.f0.startswith("chr"), hm3_bim.f0, "chr" + hm3_bim.f0)
hm3_loci = hm3_bim.select(
    locus=hl.locus(contig_38, hm3_bim.f3, reference_genome="GRCh38")
).key_by("locus")

print(f"HM3 loci loaded: {hm3_loci.count():,}")

## Filter + coalesce + checkpoint

Locus-only broadcast join (`hl.is_defined(hm3_loci[mt.locus])`), not a locus+alleles semi-join -- more robust to allele-representation differences between 1000G and AoU's calls, disambiguated by requiring biallelic SNPs rather than exact ref/alt string matching.

`naive_coalesce(384)` before the checkpoint write -- without it, the filtered MT inherits the full callset's partition count, which makes the checkpoint write far slower than it needs to be for a table this small post-filter.

In [ ]:
mt = hl.read_matrix_table(ACAF_MT_PATH)

mt = mt.filter_rows(
    hl.is_snp(mt.alleles[0], mt.alleles[1])
    & (hl.len(mt.alleles) == 2)
    & hl.is_defined(hm3_loci[mt.locus])
)

mt = mt.naive_coalesce(384)
mt = mt.checkpoint(CHECKPOINT_MT, overwrite=True)
print(f"Partitions: {mt.n_partitions()}")

## Export

Small now (post-filter), so exports straight to local scratch disk -- not back through gcsfuse.

In [ ]:
hl.export_plink(
    mt,
    EXPORT_PREFIX,
    fam_id=mt.s,
    ind_id=mt.s,
)

print(f"exported to {EXPORT_PREFIX}.{{bed,bim,fam}}")

## Relabel IDs + project

Same `--set-all-var-ids` + `--score` steps as the rest of the pipeline, now running on the small local file instead of blocking on ACAF I/O.

In [ ]:
%%bash -s "$EXPORT_PREFIX" "$PROJECT_PREFIX" "$PCA_PREFIX" "$PROJECTED_OUT"
set -e
EXPORT_PREFIX=$1
PROJECT_PREFIX_PATH=$2
PCA_PREFIX=$3
PROJECTED_OUT=$4

BUCKET_OUT_DIR=$(dirname "$PROJECT_PREFIX_PATH")
PROJECT_PREFIX_NAME=$(basename "$PROJECT_PREFIX_PATH")
cd "$(dirname "$EXPORT_PREFIX")"

plink2 \
  --bfile "$(basename "$EXPORT_PREFIX")" \
  --set-all-var-ids '@:#:$r:$a' \
  --new-id-max-allele-len 1000 \
  --make-pgen \
  --out "$PROJECT_PREFIX_NAME"

mkdir -p "$BUCKET_OUT_DIR"
cp "${PROJECT_PREFIX_NAME}".pgen "${PROJECT_PREFIX_NAME}".pvar "${PROJECT_PREFIX_NAME}".psam "$BUCKET_OUT_DIR/"

WEIGHTS="${PCA_PREFIX}.eigenvec.allele"
HEADER=$(head -1 "$WEIGHTS")
ID_COL=$(echo "$HEADER" | tr '\t' '\n' | grep -nx 'ID' | cut -d: -f1)
A1_COL=$(echo "$HEADER" | tr '\t' '\n' | grep -nx 'A1' | cut -d: -f1)
PC1_COL=$(echo "$HEADER" | tr '\t' '\n' | grep -nx 'PC1' | cut -d: -f1)
PC_LAST_COL=$(echo "$HEADER" | tr '\t' '\n' | grep -nx 'PC20' | cut -d: -f1)

plink2 \
  --pfile "$PROJECT_PREFIX_NAME" \
  --read-freq "${PCA_PREFIX}.acount" \
  --score "$WEIGHTS" "$ID_COL" "$A1_COL" header-read no-mean-imputation variance-standardize \
  --score-col-nums "${PC1_COL}-${PC_LAST_COL}" \
  --out "$(basename "$PROJECTED_OUT")"

cp "$(basename "$PROJECTED_OUT")".sscore "$BUCKET_OUT_DIR/"
head "$(basename "$PROJECTED_OUT")".sscore